# 10 · Separar las fechas antes de entrenar

Preparamos la validación de los futuros filtros de momentum y reversión. Es como reservar partes de un examen: fijamos qué datos podrá usar el modelo para aprender y con cuáles comprobaremos sus respuestas.

Antes de empezar, ejecuta **01 → 02 → 07 → 08**, en ese orden. Selecciona `.venv` como kernel y ejecuta las celdas de este cuaderno en orden. El 09 es independiente y sus asociaciones no seleccionan las características de este cuaderno. No se descargan fuentes ni se entrenan modelos.

El diseño está documentado en [el plan de validación temporal](../docs/plan_validacion_temporal.md). Trabajamos exclusivamente con 1927–1989. El periodo desde 1990 queda reservado para la evaluación posterior.

## 1. Entradas y ejemplos utilizables

Las diez variables del 07 describen información anterior a cada resultado. Las etiquetas del 08 indican si una operación aislada habría superado al efectivo. Los meses sin señal de operación conservan etiqueta ausente, **no son fallos del filtro**.

Recalculamos las transformaciones fijas con 36 observaciones anteriores, límites 1/99 y z-score con historia pasada. El recorrido cronológico usa covariables anteriores sin consultar sus etiquetas. No leemos las tablas de cribado ni ajustamos parámetros para favorecer sus cuatro asociaciones.

La regla necesita una señal, etiqueta válida y las diez variables preparadas para tener un ejemplo utilizable. Las fechas, capitales, retornos realizados y etiquetas no son predictores. La selección y cualquier ajuste adicional del modelo deberán hacerse dentro de su entrenamiento.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display

from tfm.data import project_root, read_result, save_result
from tfm.validation import (N_BLOCKS, VALIDATION_BLOCKS, PURGE_MONTHS, EMBARGO_MONTHS,
                            MODEL_COLUMNS, build_temporal_splits,
                            prepare_validation_samples, summarize_validation)

ROOT = project_root()
caracteristicas, _ = read_result(ROOT, "monthly_features")
etiquetas, informe_etiquetas = read_result(ROOT, "labelled_signals")
ejemplos, informe_preparacion = prepare_validation_samples(caracteristicas, etiquetas)
display(pd.DataFrame(informe_preparacion["sample_counts"]).rename(columns={
    "strategy": "Regla", "calendar_months": "Meses de calendario", "causal_feature_months": "Meses con preparación",
    "signal_months": "Meses con señal", "usable_examples": "Ejemplos utilizables", "positive": "Positivos", "negative": "Negativos"}))
print("Primera fecha común preparada:", informe_preparacion["first_causal_target"])
print("Diez predictores explícitos:", MODEL_COLUMNS)

## 2. Seis bloques y quince combinaciones

Dividimos primero el calendario completo en seis bloques de 126 meses. Elegimos cada pareja posible de bloques para validar y partimos de los otros cuatro para entrenar. Esto produce quince particiones, con las mismas fechas para las dos reglas y para todos los modelos.

Es **validación cruzada combinatoria purgada (CPCV)**. Durante el descubrimiento puede entrenar con bloques posteriores al que valida, como indica la guía. No representa una inversión en tiempo real, la evaluación posterior será cronológica.

Cada mes aparece cinco veces en validación. Son cinco comprobaciones del mismo mes, no cinco observaciones independientes. Este notebook aún no calcula predicciones, AUC, log-loss ni resultados de inversión.

In [ ]:
particiones, informe_calendario = build_temporal_splits()
resumen = summarize_validation(particiones, ejemplos)
display(pd.DataFrame(informe_calendario["blocks"]).rename(columns={
    "block_id": "Bloque", "first_target": "Primer objetivo", "last_target": "Último objetivo", "months": "Meses"}))
combinaciones = particiones[["split_id", "validation_block_1", "validation_block_2"]].drop_duplicates()
display(combinaciones.rename(columns={"split_id": "Partición", "validation_block_1": "Bloque de validación A",
                                     "validation_block_2": "Bloque de validación B"}).set_index("Partición"))
display(pd.Series({"Bloques": N_BLOCKS, "Bloques reservados por partición": VALIDATION_BLOCKS,
                   "Purga por frontera (meses)": PURGE_MONTHS, "Ventana de embargo posterior (meses)": EMBARGO_MONTHS},
                  name="Diseño fijado"))

## 3. Por qué dejamos separación entre fechas

Una etiqueta utiliza el resultado del mes objetivo. Evitamos que un ejemplo de entrenamiento toque el intervalo que estamos validando. Con nuestro horizonte mensual excluimos un mes antes y después de cada intervalo de validación, es una **purga conservadora**.

Después dejamos una ventana de **seis meses de embargo** desde el último mes objetivo de validación. Su primer mes ya coincide con la purga posterior, por lo que no se suma dos veces. Si los bloques de validación son adyacentes, se tratan como un solo intervalo. Las exclusiones se cuentan en meses calendario, aunque la regla no proponga operaciones durante alguno.

El gráfico muestra el papel de cada fecha. Tener color de entrenamiento no garantiza que ese mes sea un ejemplo, después aplicamos la disponibilidad de variables y la existencia de señal.

In [ ]:
colores = {"train": "#4C78A8", "validation": "#F58518", "purge": "#B279A2", "embargo": "#BDBDBD"}
rotulos = {"train": "Entrenamiento", "validation": "Validación", "purge": "Purga", "embargo": "Embargo"}
codigos = {nombre: i for i, nombre in enumerate(colores)}
matriz = particiones.pivot(index="split_id", columns="target_period", values="role")
imagen = matriz.apply(lambda column: column.map(codigos)).to_numpy(dtype=int)
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(imagen, aspect="auto", interpolation="nearest", cmap=ListedColormap(list(colores.values())), vmin=-.5, vmax=3.5)
marcas = np.arange(0, matriz.shape[1], 120)
ax.set_xticks(marcas, [matriz.columns[i][:4] for i in marcas])
ax.set_yticks(np.arange(len(matriz)), matriz.index)
ax.set(xlabel="Año del mes objetivo", ylabel="Partición", title="Las quince particiones sobre el calendario completo")
ax.legend(handles=[Patch(color=colores[role], label=rotulos[role]) for role in colores],
          loc="upper center", bbox_to_anchor=(.5, -.13), ncol=4)
plt.tight_layout()
plt.show()

## 4. Una partición y sus fronteras

Puedes elegir una partición por su número. En el ejemplo 2 se validan los bloques 1 y 3. Para el bloque 3, diciembre de 1947 se excluye por la frontera previa, julio de 1958 coincide con purga y embargo, y agosto–diciembre completan la ventana de seis meses. Enero de 1959 vuelve a estar disponible para entrenamiento.

La tabla distingue los meses de embargo adicionales de toda la ventana de embargo. El primer mes compartido con la purga se cuenta una sola vez al sumar los papeles del calendario.

In [ ]:
particion = 2
if particion not in set(particiones.split_id):
    raise ValueError("Escoge una partición entre 1 y 15.")
calendario = particiones.loc[particiones.split_id.eq(particion)].copy()
intervalos = informe_calendario["partitions"][particion - 1]["validation_intervals"]
print("Intervalos de validación de la partición elegida:")
display(pd.DataFrame(intervalos).rename(columns={"start": "Primer objetivo", "end": "Último objetivo"}))
columnas = ["target_period", "block_id", "role", "in_purge_window", "in_embargo_window"]
frontera = calendario.loc[calendario.target_period.between("1958-05", "1959-02"), columnas].copy()
frontera["role"] = frontera.role.map(rotulos)
display(frontera.rename(columns={"target_period": "Objetivo", "block_id": "Bloque", "role": "Papel",
    "in_purge_window": "En ventana de purga", "in_embargo_window": "En ventana de embargo"}).set_index("Objetivo"))

## 5. Cuántos ejemplos quedan para cada regla

Contamos por separado calendario, variables disponibles, señales y ejemplos utilizables. Los positivos y negativos corresponden a la etiqueta central. No se eliminan particiones ni se cambian las fronteras por obtener una distribución poco favorable.

Un entrenamiento con una sola clase no permite entrenar un clasificador convencional. Si falta una clase solo en validación, AUC no sería estimable. Son controles de cobertura, no puntuaciones de un modelo.

El final temporal de cada etiqueta debe coincidir con el mes objetivo. Esto no certifica cuándo se publicaron sus componentes: los dividendos reconstruidos y las revisiones de fuentes mantienen sus limitaciones. La disponibilidad efectiva para el futuro reentrenamiento cronológico deberá tratarse en esa etapa.

In [ ]:
columnas = ["strategy", "train_calendar_months", "train_feature_months", "train_signal_months", "train_examples",
            "train_positive", "train_negative", "validation_calendar_months", "validation_feature_months",
            "validation_signal_months", "validation_examples", "validation_positive", "validation_negative",
            "purge_calendar_months", "embargo_calendar_months", "embargo_window_months",
            "train_classifier_ready", "validation_auc_estimable"]
display(resumen.loc[resumen.split_id.eq(particion), columnas].set_index("strategy").T)
print("Rango de ejemplos entre las quince particiones:")
display(resumen.groupby("strategy")[["train_examples", "validation_examples"]].agg(["min", "max"]))
print("Particiones con ambas clases en entrenamiento:", int(resumen.train_classifier_ready.sum()), "de", len(resumen))
print("Particiones con ambas clases en validación:", int(resumen.validation_auc_estimable.sum()), "de", len(resumen))

## 6. Guardar las particiones

La última celda guarda o sustituye `validation_splits.parquet` y `validation_summary.parquet`, con sus JSON descriptivos en `data/processed/`. La primera tabla conserva 11.340 filas de calendario, la segunda, treinta resúmenes. No se copian las etiquetas o características en cada partición.

Regenerar las entradas de 01, 02, 07 u 08 retira estas tablas. El cribado 09 es independiente. Si falla el guardado, ejecuta de nuevo la celda completa antes de utilizar las salidas.

Con esto queda preparado el calendario. El siguiente experimento será una regresión logística con registro de variantes desde su inicio. La selección de variables y Clustered MDA usarán entrenamiento y validación interna, los bloques exteriores quedarán para evaluar. El reentrenamiento anual posterior generará señales para una cartera continua, sin reiniciar el capital cada año.

In [ ]:
informe_final = {**informe_calendario, **informe_preparacion,
    "scope": "development_partition_preparation_only",
    "parameters": informe_etiquetas["parameters"],
    "protocol": "docs/plan_validacion_temporal.md",
    "references": ["Guía del tutor 07-11-2026, sublínea 1.B", "López de Prado (2018), capítulos 7 y 12"],
    "warnings": [
        "CPCV discovery can train on later blocks, it is not a real-time investment evaluation.",
        "Only 1927–1989, no model fitting, feature selection or post-1989 evaluation.",
        "Five validation appearances per month are not independent observations.",
        "Fixed causal features use past covariates, model fitting and outcome-based selection stay within training.",
        "Monthly Shiller means and reconstructed dividends, label end is not a certified publication date.",
        "Purge and embargo do not guarantee independence of all long-window indicators.",
    ],
}
guardados = []
for nombre, tabla in {"validation_splits": particiones, "validation_summary": resumen}.items():
    salida = save_result(ROOT, nombre, tabla, informe_final)
    guardados.append({"Tabla": nombre + ".parquet", "Filas": salida["rows"]})
display(pd.DataFrame(guardados))
print("Guardado en data/processed/. No se han entrenado modelos ni modificado sus entradas.")